In [48]:
import numpy as np
import pandas as pd
from utils import OperonArgs
from os.path import join as pjoin
import matplotlib.pyplot as plt
from pathlib import Path
import csv
import re
import io

# Check data from `data` vs from `fits` repo

In [40]:
def load_data_from_data(ini_file):

    # Implementation for loading data from .ini file
    args = OperonArgs(ini_file, verbose=False)
    param_name = args.target_name
    print(f'Predicting {param_name} with version {args.version_num}, {ini_file}')

    use_dir = '/Users/deaglanbartlett/Desktop/data'
    
    path = Path(args.train_file)
    final_directory = path.parent.name
    final_name = path.name
    train_file = pjoin(use_dir, final_directory, final_name)

    path = Path(args.val_file)
    final_directory = path.parent.name
    final_name = path.name
    val_file = pjoin(use_dir, final_directory, final_name)

    df_train = pd.read_csv(train_file, sep='\t')
    df_val = pd.read_csv(val_file, sep='\t')

    # Correct the sSFR column if needed
    # Units are 1e10 yr^-1 to make O(1)
    if args.correct_ssfr:
        print('Correcting sSFR values')
        df_train['sSFR'] = df_train['SFR'] / 10.**df_train['logMstar'] * 1e10
        df_val['sSFR'] = df_val['SFR'] / 10.**df_val['logMstar'] * 1e10
        print('Median sSFR after correction:', np.median(df_train['sSFR'].values))
        print('Min sSFR after correction:', np.min(df_train['sSFR'].values[df_train['sSFR'].values > 0]))
        print('Max sSFR after correction:', np.max(df_train['sSFR'].values))
    
    return param_name, df_train, df_val


def load_data_from_fits(ini_file, length):

    args = OperonArgs(ini_file, verbose=False)
    run_name = f'{args.target_name}_{str(args.version_num)}'
    
    use_dir = '/Users/deaglanbartlett/Desktop/fits'

    out_dir = pjoin(use_dir, run_name)
    fname = pjoin(out_dir, f'{args.target_name}_train_{length}.csv')
    with open(f'{out_dir}/{run_name}_names.txt', 'r') as f:
        reader = csv.reader(f, delimiter='\t')
        names = reader.__next__()
    names[-1] = 'true'
    names.append('pred')
    df = pd.read_csv(fname, names=names, sep=r'\s+')

    fname = pjoin(out_dir, f'{args.target_name}_val_{length}.csv')
    df_val = pd.read_csv(fname, names=names, sep=r'\s+')

    return df, df_val

all_data = [
    dict(ini_file = 'conf/Av_7.ini', length = 17, old_r2 = [0.615, 0.65], do_log=True),
    dict(ini_file = 'conf/B1_9.ini', length = 14, old_r2 = [0.536, 0.529], do_log=False,),
    dict(ini_file = 'conf/B3_13.ini', length = 15, old_r2 = [0.692, 0.75], do_log=False),
    dict(ini_file = 'conf/B0_9.ini', length = 17, old_r2 = [0.927, 0.92], do_log=True, min_true = -0.5, dust_mixture='MW'),
    dict(ini_file = 'conf/B0_17.ini', length = 18, old_r2 = [0.712, 0.632], do_log=True, max_true = -2, dust_mixture='stellar'),
    dict(ini_file = 'conf/B2_15.ini', length = 18, old_r2 = [0.743, 0.809], do_log=False),
]


for data in all_data:

    ini_file = data['ini_file']
    length = data['length']
    do_log = data['do_log']
    print('')

    param_name, df_train_data, df_val_data = load_data_from_data(ini_file)
    df_train_fits, df_val_fits = load_data_from_fits(ini_file, length)

    y_train_data = df_train_data[param_name].values
    y_val_data = df_val_data[param_name].values
    y_train_fits = df_train_fits['true'].values
    y_val_fits = df_val_fits['true'].values

    if do_log:
        y_train_data = np.log10(y_train_data)
        y_val_data = np.log10(y_val_data)

    # Check data are close
    train_clos = np.isclose(y_train_data, y_train_fits, rtol=1e-5, atol=1e-8)
    val_clos = np.isclose(y_val_data, y_val_fits, rtol=1e-5, atol=1e-8)
    print(f'Param name: {param_name}, Train close: {train_clos.sum()}/{len(train_clos)}, Val close: {val_clos.sum()}/{len(val_clos)}')

    # Check sSFR columns are close
    if 'sSFR' in df_train_data.columns and 'sSFR' in df_train_fits.columns:
        sSFR_clos = np.isclose(df_train_data['sSFR'], df_train_fits['sSFR'], rtol=1e-5, atol=1e-8)
        print(f'sSFR close: {sSFR_clos.sum()}/{len(sSFR_clos)}')

print(df_train_data['dust_mixture'].value_counts())
print(df_train_fits['dust_mixture'].value_counts())



Reading from selection file: conf/selection_16.ini
Predicting Av with version 7, conf/Av_7.ini
Reading from selection file: conf/selection_16.ini
Param name: Av, Train close: 2000/2000, Val close: 1000/1000
sSFR close: 2000/2000

Reading from selection file: conf/selection_22.ini
Predicting B_1s with version 9, conf/B1_9.ini
Reading from selection file: conf/selection_22.ini
Param name: B_1s, Train close: 2000/2000, Val close: 1000/1000
sSFR close: 2000/2000

Reading from selection file: conf/selection_24.ini
Predicting B_3 with version 13, conf/B3_13.ini
Correcting sSFR values
Median sSFR after correction: 0.9962050761383203
Min sSFR after correction: 0.0005225584411379954
Max sSFR after correction: 6.9073236196721846
Reading from selection file: conf/selection_24.ini
Param name: B_3, Train close: 2000/2000, Val close: 1000/1000
sSFR close: 2000/2000

Reading from selection file: conf/selection_8.ini
Predicting B_0 with version 9, conf/B0_9.ini
Reading from selection file: conf/selec

# Get $R^2$ for training and validation

In [42]:
dust_values = {
    'MW': 0.5968,
    'SMC': 0.7193,
    'stellar': 0.3064
}

def predict_Av(f, Sigma_SFR, sini, Zgas, add_noise=True, clip=True):
    
    c = [0.428, 0.00967, 0.953, 0.00383, 1.51, 1.68, 800.0, 4.75]
    sigma = 0.221

    pred = c[0] - c[1] / (np.log10(c[2] * sini)) - c[3] / np.log10(c[4] * f) - c[5] * (c[6] * Sigma_SFR)**(-c[7] * Zgas)

    if add_noise:
        pred += np.random.normal(0, sigma, size=pred.shape)

    if clip:
        pred = np.clip(pred, -5.0, 2.0)

    return np.array(10 ** pred)

def predict_B0(f, Av, B1s, B3, dust_mixture='MW', add_noise=True, clip=True):

    if dust_mixture == 'MW':
        c = [0.662, 0.224, 0.327, 13.8, 1.53, 23.7, 0.112]
        sigma = 0.072
        pred = c[0] * B1s + c[1] * B3 - c[2] / B3 * (c[3] * B1s + c[4] * B3 + (c[5] * Av) ** (c[6] * B3))
    elif dust_mixture in ['SMC', 'stellar', 'SMC+stellar']:
        c = [0.0413, 5.79, 10.2, 3.78, 5.13, 7.2, 0.379, 0.127, 2.91]
        sigma = 0.216
        pred = c[0] * B3 * (c[1] * Av + (c[2] * Av)**(c[3]*B1s) - c[4] * np.log10(c[5] * B3) + (c[6] * f)**(-c[7] * B3)) - c[8]
    else:
        raise ValueError(f'Unknown dust mixture: {dust_mixture}')
    
    if add_noise:
        pred += np.random.normal(0, sigma, size=pred.shape)

    if clip:
        pred = np.clip(pred, -5.0, 2.0)

    return np.array(10 ** pred)

def predict_B1s(f, Av, Zgas, sini, log10Mstar, add_noise=True, clip=True):

    c = [0.0324, 25.5, 2.36, 0.00411, 1.95]
    sigma = 0.058
    pred = c[0] / Av * ((c[1] * Zgas)**(c[2] * sini) - c[3] * log10Mstar / np.log10(c[4] * f))

    if add_noise:
        pred += np.random.normal(0, sigma, size=pred.shape)

    if clip:
        pred = np.clip(pred, -1.0, 1.0)

    return np.array(pred)

def predict_B2s(f, B0, B1s, add_noise=True, clip=True):

    c = [0.264, 0.129, 0.00351, 0.776, 1.29, 5.93, 179.0]
    sigma = 0.381
    pred = c[0] * B0 + c[1] - f * (c[2] * B0**(-c[3]) + c[4] * B1s * (c[5] * B1s + np.log10(c[6] * B0)))

    if add_noise:
        pred += np.random.normal(0, sigma, size=pred.shape)

    if clip:
        pred = np.clip(pred, -5.0, 5.0)

    return np.array(pred)

def predict_B3(f, Av, Zgas, sini, sSFR, add_noise=True, clip=True):

    c = [2.1, 0.83, 51.1, 21.2, 0.97, 3.52, 0.0417, 0.17]
    sigma = 0.728
    pred = - c[0] - c[1] * sini + c[2] * Zgas * (c[3] * Av) ** (- c[4] * sSFR) + c[5] * (c[6] * Av) ** (-c[7] * f)

    if add_noise:
        pred += np.random.normal(0, sigma, size=pred.shape)

    if clip:
        pred = np.clip(pred, 3.e-2, 10.0)

    return np.array(pred)


def get_predictions(df, target, dust_mixture='MW', add_noise=True, clip=True):

    if target == 'Av':
        return predict_Av(f=df['xi_dust'], 
                        Sigma_SFR=df['Sigma_SFR'], 
                        sini=df['inclination_sin'], 
                        Zgas=df['Zgas'], add_noise=add_noise, clip=clip)
    if target == 'B_0':
        return predict_B0(f=df['xi_dust'], 
                        Av=df['Av'], 
                        B1s=df['B_1s'], 
                        B3=df['B_3'],
                        dust_mixture=dust_mixture,
                        add_noise=add_noise, clip=clip)
    if target == 'B_1s':
        return predict_B1s(f=df['xi_dust'], 
                        Av=df['Av'],
                        Zgas=df['Zgas'], 
                        sini=df['inclination_sin'], 
                        log10Mstar=df['logMstar'], add_noise=add_noise, clip=clip)
    if target == 'B_2s':
        return predict_B2s(f=df['xi_dust'], B0=df['B_0'], B1s=df['B_1s'], add_noise=add_noise, clip=clip)
    if target == 'B_3':
        return predict_B3(f=df['xi_dust'], 
                         Av=df['Av'], 
                         Zgas=df['Zgas'], 
                         sini=df['inclination_sin'],
                         sSFR=df['sSFR'], add_noise=add_noise, clip=clip)


    raise ValueError(f'Unknown target: {target}')

In [73]:
def replace_val(df_val,):

    # Select galaxy_ids in validation set if their dust mixture is MW
    selected_galaxy_ids = df_val[df_val['dust_mixture'] == 'MW']['galaxy_id'].values

    new_fname = '/Users/deaglanbartlett/Desktop/iob_codes_plus_Acurve_galprop_MW_noBUG_full.dat'

    # Only load the galaxy_id column to find the rows corresponding to the selected galaxy_ids
    gid_df = pd.read_csv(new_fname, sep=r'\s+', usecols=["galaxy_id"])

    # Find the rows corresponding to the selected galaxy_ids
    selected_rows = gid_df[gid_df['galaxy_id'].isin(selected_galaxy_ids)].index

    # Now load full data for these rows
    selected_set = set(selected_rows)
    out_lines = []
    with open(new_fname, "r") as fh:
        header = next(fh)
        # Remove any part of header between square brackets, 
        # as this is not a valid column name and causes problems for pandas
        header = re.sub(r'\[.*?\]', '', header)
        for i, line in enumerate(fh, start=1):
            if i-1 in selected_set:
                out_lines.append(line)
    data = "".join([header] + out_lines)
    new_df = pd.read_csv(io.StringIO(data), sep=r'\s+')

    # Now replace the rows of df_val with the new data from new_df, matching on galaxy_id and los and
    # checking the dust_mixture is MW in df_val
    mw_mask = df_val['dust_mixture'] == 'MW'
    key_cols = ['galaxy_id', 'los']
    common_cols = [c for c in df_val.columns if c in new_df.columns and c not in key_cols]

    # Drop duplicates on key columns to prevent fan-out during merge
    new_df_sub = new_df[key_cols + common_cols].drop_duplicates(subset=key_cols)
    new_df_val = df_val.copy()
    df_val.loc[mw_mask, common_cols] = (
        df_val.loc[mw_mask, key_cols]
        .merge(new_df_sub, on=key_cols, how='left')[common_cols]
        .values
    )

    # Print how many rows were replaced
    replaced_rows = mw_mask.sum()
    print(f'Replaced {replaced_rows} rows in validation set with new data')

    return new_df_val


def load_data(ini_file):
    # Implementation for loading data from .ini file
    args = OperonArgs(ini_file,)
    param_name = args.target_name
    print(f'Predicting {param_name} with version {args.version_num}, {ini_file}')

    use_dir = '/Users/deaglanbartlett/Desktop/data'
    
    path = Path(args.train_file)
    final_directory = path.parent.name
    final_name = path.name
    train_file = pjoin(use_dir, final_directory, final_name)

    path = Path(args.val_file)
    final_directory = path.parent.name
    final_name = path.name
    val_file = pjoin(use_dir, final_directory, final_name)

    df_train = pd.read_csv(train_file, sep='\t')
    df_val = pd.read_csv(val_file, sep='\t')

    # MW         783
    # stellar    631
    # SMC        586

    # dust_mixture
    # 0.5968    783
    # 0.3064    631
    # 0.7193    586

    dust_mixture_dict = {
        'MW': 0.5968,
        'SMC': 0.7193,
        'stellar': 0.3064
    }

    # Add new columns for the dust values
    df_train['xi_dust'] = df_train['dust_mixture'].map(dust_mixture_dict)
    df_val['xi_dust'] = df_val['dust_mixture'].map(dust_mixture_dict)

    # Correct the sSFR column if needed
    # Units are 1e10 yr^-1 to make O(1)
    if args.correct_ssfr:
        df_train['sSFR'] = df_train['SFR'] / 10.**df_train['logMstar'] * 1e10
        df_val['sSFR'] = df_val['SFR'] / 10.**df_val['logMstar'] * 1e10
    
    return param_name, df_train, df_val


def get_r2(true, pred):
    return 1 - np.sum((true - pred) ** 2) / np.sum((true - np.mean(true)) ** 2)

all_data = [
    dict(ini_file = 'conf/Av_7.ini', length = 17, old_r2 = [0.615, 0.65], do_log=True),
    dict(ini_file = 'conf/B1_9.ini', length = 14, old_r2 = [0.537, 0.529], do_log=False,),
    dict(ini_file = 'conf/B3_13.ini', length = 15, old_r2 = [0.692, 0.75], do_log=False),
    dict(ini_file = 'conf/B0_9.ini', length = 17, old_r2 = [0.931, 0.927], do_log=True, min_true = -0.5, dust_mixture='MW'),
    dict(ini_file = 'conf/B0_17.ini', length = 18, old_r2 = [0.712, 0.632], do_log=True, max_true = -2, dust_mixture='SMC+stellar'),
    dict(ini_file = 'conf/B2_15.ini', length = 18, old_r2 = [0.743, 0.809], do_log=False),
]



for data in all_data:
    
    param_name, df, df_val = load_data(data['ini_file'],)
    new_df_val = replace_val(df_val)
    
    if 'dust_mixture' in data:
        pred_train = get_predictions(df, target=param_name, add_noise=False, dust_mixture=data['dust_mixture'], clip=False)
        pred_val = get_predictions(df_val, target=param_name, add_noise=False, dust_mixture=data['dust_mixture'], clip=False)
        new_pred_val = get_predictions(new_df_val, target=param_name, add_noise=False, dust_mixture=data['dust_mixture'], clip=False)
    else:
        pred_train = get_predictions(df, target=param_name, add_noise=False, clip=False)
        pred_val = get_predictions(df_val, target=param_name, add_noise=False, clip=False)
        new_pred_val = get_predictions(new_df_val, target=param_name, add_noise=False, clip=False)

    if data['do_log']:
        pred_train = np.log10(pred_train)
        pred_val = np.log10(pred_val)
        true_train = np.log10(df[param_name])
        true_val = np.log10(df_val[param_name])
        new_true_val = np.log10(new_df_val[param_name])
        new_pred_val = np.log10(new_pred_val)
    else:
        true_train = df[param_name]
        true_val = df_val[param_name]
        new_true_val = new_df_val[param_name]

    if 'min_true' in data:
        m = true_train >= data['min_true']
        true_train = true_train[m]
        pred_train = pred_train[m]
        m = true_val >= data['min_true']
        true_val = true_val[m]
        pred_val = pred_val[m]
        m = new_true_val >= data['min_true']
        new_true_val = new_true_val[m]
        new_pred_val = new_pred_val[m]
    if 'max_true' in data:
        m = true_train <= data['max_true']
        true_train = true_train[m]
        pred_train = pred_train[m]
        m = true_val <= data['max_true']
        true_val = true_val[m]
        pred_val = pred_val[m]
        m = new_true_val <= data['max_true']
        new_true_val = new_true_val[m]
        new_pred_val = new_pred_val[m]

    new_r2 = [
        get_r2(true_train, pred_train),
        get_r2(true_val, pred_val)
    ]
    # Round
    new_r2 = np.abs([round(r, 3) for r in new_r2])
    good_matches = np.isclose(new_r2, data['old_r2'], rtol=1e-3, atol=1e-3)
    print('Old r2 good:', good_matches)

    # Get new r2 with the new validation set
    new_r2[-1] = get_r2(new_true_val, new_pred_val)
    new_r2 = np.abs([round(r, 3) for r in new_r2])
    print(f"New r2: {new_r2}")



Reading from configuration file: conf/Av_7.ini
Reading from selection file: conf/selection_16.ini
Predicting Av with version 7, conf/Av_7.ini
Replaced 375 rows in validation set with new data
Old r2 good: [ True  True]
New r2: [0.615 0.65 ]

Reading from configuration file: conf/B1_9.ini
Reading from selection file: conf/selection_22.ini
Predicting B_1s with version 9, conf/B1_9.ini
Replaced 374 rows in validation set with new data
Old r2 good: [ True  True]
New r2: [0.537 0.529]

Reading from configuration file: conf/B3_13.ini
Reading from selection file: conf/selection_24.ini
Predicting B_3 with version 13, conf/B3_13.ini
Replaced 374 rows in validation set with new data
Old r2 good: [ True False]
New r2: [0.692 0.75 ]

Reading from configuration file: conf/B0_9.ini
Reading from selection file: conf/selection_8.ini
Predicting B_0 with version 9, conf/B0_9.ini
Replaced 1000 rows in validation set with new data
Old r2 good: [ True False]
New r2: [0.931 0.927]

Reading from configurati

In [27]:
def get_relative_paths(root_dir):
    root = Path(root_dir)

    return [
        str(file.relative_to(root))
        for file in root.rglob("*")
        if file.is_file()
    ]

paths = get_relative_paths("/Users/deaglanbartlett/Desktop/iob-attenuation/gal_props/data")

new_path = '/mnt/users/deaglan/symbolic_regression/iob-attenuation/gal_props/data'

paths.sort()
paths = [f"{new_path}/{p}" for p in paths if p.endswith('.txt')]

print('Number of files:', len(paths))

# for p in paths:
#     print(p)

Number of files: 96
